In [1]:
import pandas as pd
import numpy as np

# Load raw data
raw_path = "../data/raw/Online Retail.xlsx"
df = pd.read_excel(raw_path)

# Standardize columns
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

# Drop rows missing key identifiers
df = df.dropna(subset=["customerid", "description"])

# Types
df["customerid"] = df["customerid"].astype(int)
df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")

# Remove cancellations (invoice starts with 'C') and invalid values
df["is_cancelled"] = df["invoiceno"].astype(str).str.startswith("C")
df = df[(df["quantity"] > 0) & (df["unitprice"] > 0) & (~df["is_cancelled"])]

# Feature engineering
df["revenue"] = df["quantity"] * df["unitprice"]
df["date"] = df["invoicedate"].dt.date
df["month"] = df["invoicedate"].dt.to_period("M").astype(str)
df["weekday"] = df["invoicedate"].dt.day_name()

# Rename fields for clarity
df = df.rename(columns={
    "invoiceno": "invoice_no",
    "stockcode": "sku",
    "unitprice": "unit_price",
    "customerid": "customer_id",
    "invoicedate": "invoice_date"
})

# Remove extreme revenue outliers (keeps trends realistic)
upper_rev = df["revenue"].quantile(0.995)
df = df[df["revenue"] <= upper_rev]

# Save processed dataset
out_path = "../data/processed/online_retail_cleaned.csv"
df.to_csv(out_path, index=False)

df.head(), df.shape


(  invoice_no     sku                          description  quantity  \
 0     536365  85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
 1     536365   71053                  WHITE METAL LANTERN         6   
 2     536365  84406B       CREAM CUPID HEARTS COAT HANGER         8   
 3     536365  84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
 4     536365  84029E       RED WOOLLY HOTTIE WHITE HEART.         6   
 
          invoice_date  unit_price  customer_id         country  is_cancelled  \
 0 2010-12-01 08:26:00        2.55        17850  United Kingdom         False   
 1 2010-12-01 08:26:00        3.39        17850  United Kingdom         False   
 2 2010-12-01 08:26:00        2.75        17850  United Kingdom         False   
 3 2010-12-01 08:26:00        3.39        17850  United Kingdom         False   
 4 2010-12-01 08:26:00        3.39        17850  United Kingdom         False   
 
    revenue        date    month    weekday  
 0    15.30  2010-12-01  2010-12